# Sugar Spike Predictor — Analysis Notebook

Post-meal blood glucose prediction (~2 hours after eating) using **population-specific** models:

- **Diabetic** cohort (A1c ≥ 6.5) → XGBoost
- **Non-diabetic** cohort → HistGradientBoosting

**Live demo:** [https://sugar-spike-predictor-92g4.onrender.com/](https://sugar-spike-predictor-92g4.onrender.com/)

**Repo:** [HarishKarthickS/sugar-spike-predictor](https://github.com/HarishKarthickS/sugar-spike-predictor)

> Not medical advice. For research and education only.


## 1. Setup

Run from the repository root (or open this notebook from `notebooks/` — paths auto-adjust). Feature builders match production code in `app/features.py`.


In [ ]:
from pathlib import Path
import sys

import joblib
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from app.config import Config
from app.features import adjust_prediction, build_feature_row, estimate_accuracy, time_category

DATA = ROOT / "data"
ARTIFACTS = ROOT / "artifacts"
MEAL_FEATURES = DATA / "meal_features.csv"

print("ROOT:", ROOT)
print("meal_features exists:", MEAL_FEATURES.exists())
print("diabetic model:", Config.DIABETIC_MODEL_PATH.exists())
print("non-diabetic model:", Config.NON_DIABETIC_MODEL_PATH.exists())


## 2. Problem

Given personal health metrics (age, BMI, A1c, insulin, heart rate, current glucose / trend) and meal nutrition
(calories, carbs, protein, fat, fiber), estimate glucose **about two hours** after the meal.

A single global regressor underfits cohort differences, so training splits subjects by diabetic status and fits separate pipelines.


## 3. Data

Expected local training inputs (gitignored — CGM/bio data stays private):

1. `data/bio.csv` + `data/CGMacros-*.csv`
2. `python training/data_preparation.py` → `data/merged_cgm_bio_data.csv`
3. `python training/train_baseline.py` → `data/meal_features.csv`
4. `python training/train_specialized.py` → pickles in `artifacts/`

The deployed app does **not** need raw CSVs; it loads `artifacts/*.pkl` shipped with the repo.


In [ ]:
if MEAL_FEATURES.exists():
    meals = pd.read_csv(MEAL_FEATURES)
    print("shape:", meals.shape)
    print("columns:", meals.columns.tolist()[:25], "...")
    if "is_diabetic" in meals.columns:
        print(meals["is_diabetic"].value_counts(dropna=False))
    print(meals.head(3))
else:
    print(
        "Skip: data/meal_features.csv not found. "
        "Place training CSVs under data/ and run the training scripts to reproduce evaluation."
    )


## 4. Feature engineering (same as production)

Engineered signals used by both training and the Flask inference path:

- glycemic load, carb–insulin ratio, fat+protein–to–carb
- BMI×insulin, age×A1c, glucose variability / momentum
- meal energy density, time-of-day category


In [ ]:
demo_person = {
    "age": 45,
    "gender": "Female",
    "bmi": 28.0,
    "a1c": 6.7,
    "fasting_glucose": 130,
    "insulin_level": 15.0,
    "heart_rate": 75,
}
demo_meal = {
    "meal_type": "Lunch",
    "calories": 600,
    "carbs": 60,
    "protein": 25,
    "fat": 20,
    "fiber": 8,
}

features = build_feature_row(demo_person, demo_meal, current_glucose=120, glucose_trend=0.2, hour=12)
engineered = [
    "glycemic_load",
    "carb_insulin_ratio",
    "fat_protein_to_carb",
    "bmi_insulin",
    "time_category",
    "glucose_variability",
    "meal_energy_density",
    "glucose_momentum",
    "age_a1c",
]
pd.Series({k: features[k] for k in engineered})


In [ ]:
print("time buckets:", {h: time_category(h) for h in [2, 7, 12, 15, 20]})


## 5. Models

Specialized training (`training/train_specialized.py`):

| Cohort | Model | Notes |
| --- | --- | --- |
| Diabetic | XGBoost regressor | Tuned for larger glucose swings |
| Non-diabetic | HistGradientBoosting | Typically smoother responses |

Each model is an sklearn `Pipeline`: impute → scale / one-hot → `SelectFromModel` → regressor.


## 6. Metrics

Approximate **held-out test RMSE** used by the app for prediction ranges (`app/config.py`):

| Cohort | Approx. test RMSE |
| --- | --- |
| Diabetic (XGBoost) | **~28.5 mg/dL** |
| Non-diabetic (HistGradientBoosting) | **~18.7 mg/dL** |

Earlier single-model baselines were roughly ~34 / ~22 mg/dL (comments from the specialized-app path).

When `data/meal_features.csv` is available, the next cell recomputes RMSE / MAE / R² with the same cohort split.


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

REPORTED = {
    "diabetic": {"rmse": Config.DIABETIC_RMSE},
    "non_diabetic": {"rmse": Config.NON_DIABETIC_RMSE},
}
print("App-reported approximate RMSE:", REPORTED)

if not MEAL_FEATURES.exists():
    print("Skip recompute: meal_features.csv missing.")
elif not Config.DIABETIC_MODEL_PATH.exists() or not Config.NON_DIABETIC_MODEL_PATH.exists():
    print("Skip recompute: model pickles missing under artifacts/.")
else:
    df = pd.read_csv(MEAL_FEATURES)
    drop_cols = [
        "target_glucose",
        "meal_id",
        "meal_timestamp",
        "subject_id",
        "actual_prediction_minutes",
        "is_diabetic",
    ]
    drop_cols = [c for c in drop_cols if c in df.columns]

    def eval_cohort(frame, model_path, label):
        X = frame.drop(columns=drop_cols, errors="ignore")
        y = frame["target_glucose"]
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
        model = joblib.load(model_path)
        pred = model.predict(X_test)
        rmse = float(np.sqrt(mean_squared_error(y_test, pred)))
        mae = float(mean_absolute_error(y_test, pred))
        r2 = float(r2_score(y_test, pred))
        print(f"{label}: n_test={len(y_test)}  RMSE={rmse:.2f}  MAE={mae:.2f}  R2={r2:.3f}")
        return {"rmse": rmse, "mae": mae, "r2": r2, "n_test": int(len(y_test))}

    metrics = {}
    if "is_diabetic" in df.columns:
        diabetic = df[df["is_diabetic"] == True]
        non_diabetic = df[df["is_diabetic"] == False]
        if len(diabetic):
            metrics["diabetic"] = eval_cohort(diabetic, Config.DIABETIC_MODEL_PATH, "diabetic")
        if len(non_diabetic):
            metrics["non_diabetic"] = eval_cohort(
                non_diabetic, Config.NON_DIABETIC_MODEL_PATH, "non_diabetic"
            )
    metrics


## 7. Inference demo (production path)

Load shipped pickles and run the same feature builder the Flask app uses.


In [ ]:
def predict_demo(person, meal, current_glucose, glucose_trend):
    is_diabetic = float(person.get("a1c", 0)) >= Config.A1C_DIABETIC_THRESHOLD
    path = Config.DIABETIC_MODEL_PATH if is_diabetic else Config.NON_DIABETIC_MODEL_PATH
    if not path.exists():
        raise FileNotFoundError(path)
    model = joblib.load(path)
    row = build_feature_row(person, meal, current_glucose, glucose_trend)
    raw = float(model.predict(pd.DataFrame([row]))[0])
    adjusted = adjust_prediction(raw, row)
    rmse = Config.DIABETIC_RMSE if is_diabetic else Config.NON_DIABETIC_RMSE
    return {
        "cohort": "diabetic" if is_diabetic else "non_diabetic",
        "prediction_mg_dL": round(adjusted, 1),
        "expected_range": (round(adjusted - rmse, 1), round(adjusted + rmse, 1)),
        "approx_accuracy": round(estimate_accuracy(rmse, row, is_diabetic), 1),
    }

if Config.DIABETIC_MODEL_PATH.exists() and Config.NON_DIABETIC_MODEL_PATH.exists():
    diabetic_example = predict_demo(demo_person, demo_meal, 120, 0.2)
    non_diabetic_person = {**demo_person, "a1c": 5.2}
    non_diabetic_example = predict_demo(non_diabetic_person, demo_meal, 95, 0.1)
    print("Diabetic example:", diabetic_example)
    print("Non-diabetic example:", non_diabetic_example)
else:
    print("Skip inference demo: artifacts missing.")


## 8. Reproduce training

```bash
pip install -r requirements.txt
python training/data_preparation.py
python training/train_baseline.py
python training/train_specialized.py
```

Web app: `python run.py` → http://localhost:5000

Live: https://sugar-spike-predictor-92g4.onrender.com/
